In [71]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
from GNB.gaussian_naive_bayes import GNB
from LogisticRegresssion.LogisticRegression import LogReg
import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



In [72]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [73]:
y_train_bin = (y_train == 6).astype(int)
y_test_bin  = (y_test == 6).astype(int)


In [74]:
# Flatten
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [75]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [76]:
def compute_pixel_variance(X):
    mean = np.mean(X, axis=0)
    return np.mean((X - mean) ** 2, axis=0)

def variance_threshold(X, threshold=1e-4):
    variances = compute_pixel_variance(X)
    mask = variances > threshold
    return X[:, mask], mask

X_train_clean, mask = variance_threshold(X_train)
X_test_clean = X_test[:, mask]

print("After variance filtering:", X_train_clean.shape)


After variance filtering: (60000, 623)


In [77]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [78]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [79]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [80]:
print(X_test_hog_pca.shape)

(10000, 50)


In [81]:
#Decision Tree results
X_train_hog_pca_dt = X_train_hog_pca[10000 : ]
dt = DecisionTree(maxDepth=12 , minSampleLeafs=5 , minSamplesSplit=10 , criterion='entropy' ,maxFeatures= 'sqrt' )


dt.fit(X_train_hog_pca , y_train_bin)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin, predictions,
    target_names=["Not 6", "Is 6"]
))



              precision    recall  f1-score   support

       Not 6       0.99      0.99      0.99      9042
        Is 6       0.92      0.91      0.91       958

    accuracy                           0.98     10000
   macro avg       0.96      0.95      0.95     10000
weighted avg       0.98      0.98      0.98     10000



In [82]:
gnb = GNB()

gnb.gaussian_naive_train(X_train_hog_pca, y_train_bin)
predictions = gnb.predict(X_test_hog_pca)

print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

              precision    recall  f1-score   support

       Not 6       0.99      1.00      0.99      9042
        Is 6       0.99      0.87      0.93       958

    accuracy                           0.99     10000
   macro avg       0.99      0.94      0.96     10000
weighted avg       0.99      0.99      0.99     10000



In [83]:
lg = LogReg(max_iterations=1000 , learning_rate=0.1 , threshold=0.5)
lg.fit(X_train_hog_pca , y_train_bin)
predictions = lg.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

Iteration 0, Loss: 0.6931471805599453
Iteration 10, Loss: 0.5204836192257609
Iteration 20, Loss: 0.4145701068224873
Iteration 30, Loss: 0.3451067925918461
Iteration 40, Loss: 0.29667728337337074
Iteration 50, Loss: 0.2611893910700289
Iteration 60, Loss: 0.23413069729872668
Iteration 70, Loss: 0.21283115605013367
Iteration 80, Loss: 0.1956263733916715
Iteration 90, Loss: 0.18143147354935074
Iteration 100, Loss: 0.16951182700195705
Iteration 110, Loss: 0.15935343490202142
Iteration 120, Loss: 0.15058637903856065
Iteration 130, Loss: 0.14293783953189243
Iteration 140, Loss: 0.1362022649777986
Iteration 150, Loss: 0.13022185565850214
Iteration 160, Loss: 0.124873448733407
Iteration 170, Loss: 0.12005949273102157
Iteration 180, Loss: 0.11570170203077074
Iteration 190, Loss: 0.11173650886200572
Iteration 200, Loss: 0.10811174646345734
Iteration 210, Loss: 0.10478419169465916
Iteration 220, Loss: 0.10171771810835967
Iteration 230, Loss: 0.09888188955097539
Iteration 240, Loss: 0.0962508763097

In [84]:
y_train_bin = np.where(y_train == 6, 1, -1)
y_test_bin  = np.where(y_test == 6, 1, -1)

In [85]:
svm = LinearSVMScartch(
    C=1.0,
    learning_rate=0.0001,
    n_epochs=100,
    batch_size=128,
    use_class_weights=True,
    random_state=42
)
svm.fit(X_train_hog_pca , y_train_bin)
predictions = svm.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

              precision    recall  f1-score   support

       Not 6       1.00      0.99      1.00      9042
        Is 6       0.93      0.99      0.95       958

    accuracy                           0.99     10000
   macro avg       0.96      0.99      0.98     10000
weighted avg       0.99      0.99      0.99     10000

